In [1]:
"""
Reproduces Fig. 2, Fig. 3 and (part of) Fig. 4 from Ratnakar (2026),
"Thermodynamics and equilibrium thermochemistry of ortho- and
para-hydrogen...", Int. J. Hydrogen Energy 245, 155702 -- the ideal-state
(quantum-mechanical) part of the paper, Sections 3.1-3.2.

IMPORTANT CAVEAT (see also printed note and figure captions):
The paper's absolute H0, S0, G0, U0 depend on reference constants
(Href, Sref, Uref, Gref) that are not given in the paper's main text
(likely in the Supplementary Material, which I don't have). This script
sets them all to 0 for every isomer -- consistent with each other (so
DIFFERENCES between isomers, Fig.2-style, are exact and reference-free),
but the ABSOLUTE curves (H0/R, S0/R, G0/R, U0/R vs T) have an arbitrary,
unverified vertical offset. Their SHAPE/slope is still meaningful.
"""

import numpy as np
import matplotlib.pyplot as plt
from h2_thermo_final import R, ideal_properties, Q_partition

OUTDIR = "figures"

# ---------------------------------------------------------------------
# Common temperature grids (paper uses ~0-600K linear for Fig.2-3,
# log-scale ~10-500K for Fig.4)
# ---------------------------------------------------------------------
T_lin = np.linspace(5, 600, 400)          # for Fig.2, Fig.3
T_log = np.geomspace(10, 500, 300)        # for Fig.4

species_list = ["p", "o", "eq", "n"]
species_label = {"p": "pH2", "o": "oH2", "eq": "eqH2", "n": "nH2"}

# Standard-state ideal-gas molar volume (1 bar, given T) -- only used for
# the ABSOLUTE entropy/Gibbs-energy curves (Eq.9-10); the DIFFERENCE plots
# (Fig.2-style) are independent of this choice (the V terms cancel).
P0 = 1.0e5  # Pa

def V_std(T):
    return R * T / P0

# ---------------------------------------------------------------------
# Compute ideal-state properties for every species, over T_lin
# ---------------------------------------------------------------------
props = {sp: {"H": [], "S": [], "G": [], "U": [], "Cp": []} for sp in species_list}
for T in T_lin:
    V = V_std(T)
    for sp in species_list:
        d = ideal_properties(T, V, sp)
        for key in ("H", "S", "G", "U", "Cp"):
            props[sp][key].append(d[key])
for sp in species_list:
    for key in props[sp]:
        props[sp][key] = np.array(props[sp][key])


# =======================================================================
# FIGURE "2_absolute": H0/R, S0/R, G0/R, U0/R vs T -- ABSOLUTE VALUES
# =======================================================================
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
panel_info = [
    ("H", "H^0/R  [K]", axes[0, 0], "(a)"),
    ("S", "S^0/R", axes[0, 1], "(b)"),
    ("G", "G^0/R  [K]", axes[1, 0], "(c)"),
    ("U", "U^0/R  [K]", axes[1, 1], "(d)"),
]
colors = {"p": "tab:red", "o": "tab:blue", "eq": "tab:green", "n": "black"}
for key, ylabel, ax, tag in panel_info:
    for sp in species_list:
        ax.plot(T_lin, props[sp][key] / R, label=species_label[sp], color=colors[sp])
    ax.set_xlabel("T, K")
    ax.set_ylabel(ylabel)
    ax.set_title(tag, loc="left", fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("Ideal-state ABSOLUTE properties vs T (Href=Sref=Uref=Gref=0 -- offset is arbitrary, see caveat)",
             fontsize=10, color="darkred")
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{OUTDIR}/fig2_absolute_ideal_properties.png", dpi=150)
plt.close(fig)


# =======================================================================
# FIGURE "2_differences": mirrors the paper's Fig.2 exactly (a-d)
# =======================================================================
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
diff_info = [
    ("H", "\u0394H^0/R  [K]", axes[0, 0], "(a)"),
    ("S", "\u0394S^0/R", axes[0, 1], "(b)"),
    ("G", "\u0394G^0/R  [K]", axes[1, 0], "(c)"),
    ("U", "\u0394U^0/R  [K]", axes[1, 1], "(d)"),
]
diff_pairs = [
    ("o", "p", "tab:blue",  "oH2 - pH2"),
    ("n", "p", "black",     "nH2 - pH2"),
    ("eq","p", "tab:green", "eqH2 - pH2"),
    ("n", "eq","tab:red",   "nH2 - eqH2"),
]
for key, ylabel, ax, tag in diff_info:
    for sp1, sp2, color, label in diff_pairs:
        ax.plot(T_lin, (props[sp1][key] - props[sp2][key]) / R, label=label, color=color)
    ax.set_xlabel("T, K")
    ax.set_ylabel(ylabel)
    ax.set_title(tag, loc="left", fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("Ideal-state property DIFFERENCES between hydrogen isomers vs T\n"
             "(reproduces paper's Fig. 2 -- reference-independent, exact)", fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(f"{OUTDIR}/fig2_differences_ideal_properties.png", dpi=150)
plt.close(fig)


# =======================================================================
# FIGURE "3": Cp0/R vs T (reproduces paper's Fig. 3)
# =======================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
for sp, style in [("o", "tab:blue"), ("p", "tab:red"), ("n", "black"), ("eq", "tab:green")]:
    ax.plot(T_lin, props[sp]["Cp"] / R, label=species_label[sp], color=style)
ax.axhline(2.5, color="gray", ls=":", lw=1, label="5/2 R (low-T limit)")
ax.axhline(3.5, color="gray", ls="--", lw=1, label="7/2 R (high-T limit)")
ax.set_xlabel("T, K")
ax.set_ylabel(r"$C_P^0/R$")
ax.set_title("Ideal-state isobaric heat capacity vs T (reproduces paper's Fig. 3)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUTDIR}/fig3_Cp0_ideal.png", dpi=150)
plt.close(fig)


# =======================================================================
# FIGURE "4": equilibrium para-H2 mole fraction & heat of conversion vs T
# =======================================================================
y_eq = np.array([Q_partition(T, "p") / (Q_partition(T, "p") + Q_partition(T, "o")) for T in T_log])
Hn_minus_Hp = np.array([
    (0.25*ideal_properties(T, 1.0, "p")["H"] + 0.75*ideal_properties(T, 1.0, "o")["H"])
    - ideal_properties(T, 1.0, "p")["H"]
    for T in T_log
])

fig, ax1 = plt.subplots(figsize=(7, 5.5))
ax1.plot(T_log, y_eq, color="tab:red", label=r"$y_{eq}$ (para mole fraction)")
ax1.set_xscale("log")
ax1.set_xlabel("T, K")
ax1.set_ylabel(r"$y_{eq}$", color="tab:red")
ax1.axhline(0.25, color="tab:red", ls=":", lw=1)
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(T_log, Hn_minus_Hp, color="tab:blue", label=r"$H_n^0-H_p^0$")
ax2.set_ylabel(r"$H_n^0 - H_p^0$  [J/mol]", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")

ax1.set_title("Equilibrium para-H2 fraction and heat of conversion vs T\n"
              "(reproduces paper's Fig. 4 -- no NIST monograph overlay available)")
fig.tight_layout()
fig.savefig(f"{OUTDIR}/fig4_equilibrium_composition.png", dpi=150)
plt.close(fig)

print("Done. Saved:")
print("  fig2_absolute_ideal_properties.png   (H0,S0,G0,U0 vs T -- ARBITRARY vertical offset)")
print("  fig2_differences_ideal_properties.png (Delta-H0,S0,G0,U0 vs T -- exact, matches paper's Fig.2)")
print("  fig3_Cp0_ideal.png                    (Cp0/R vs T -- matches paper's Fig.3)")
print("  fig4_equilibrium_composition.png      (y_eq and Hn-Hp vs T -- matches paper's Fig.4, no data overlay)")

Done. Saved:
  fig2_absolute_ideal_properties.png   (H0,S0,G0,U0 vs T -- ARBITRARY vertical offset)
  fig2_differences_ideal_properties.png (Delta-H0,S0,G0,U0 vs T -- exact, matches paper's Fig.2)
  fig3_Cp0_ideal.png                    (Cp0/R vs T -- matches paper's Fig.3)
  fig4_equilibrium_composition.png      (y_eq and Hn-Hp vs T -- matches paper's Fig.4, no data overlay)
